# 파지 강도 동적 계산 — v2

**담당**: 송다현, 정수진 (동적 강도 파지)

v1(`grasp.ipynb`) 재작성본. 원본은 그대로 뒀습니다.
근거: [`REVIEW.md`](REVIEW.md), 리뷰 반영: [`grasp_v2_문의점.txt`](grasp_v2_문의점.txt)

## 핵심

**로봇이 무게를 잽니다.** 유일한 실측값은 `actual_motor_torque` (`read_data_rt`) 입니다.
`get_tool_force()` / `get_workpiece_weight()` 는 **구조적으로 항상 0** 입니다
(E0509 에는 토크 센서도 F/T 센서도 없어서, `raw_joint_torque` 가 사실 `gravity_torque` 입니다).

```
① 상자 폭에 맞춰 살짝 집는다 → ② 무게를 잰다 → ③ 힘을 계산한다
                              → ④ 그 힘으로 다시 조인다 → ⑤ 들어서 확인 → ⑥ 옮긴다
```

## ⚠️ 기준선은 그리퍼 위치에 따라 달라집니다

빈 그리퍼를 **열었다 닫는 것만으로 토크가 0.137 Nm 바뀝니다** (≈ 43 g 상당).
손가락이 움직이면 무게중심이 이동하기 때문입니다.

**상자 크기가 다르면 손가락이 멈추는 위치도 다릅니다.** 그래서 상자마다 기준선이 달라집니다.

우리 그리퍼는 **현재 위치를 읽을 수 없으므로**(`GripperCmd.srv` = 제어 전용, 피드백 없음),
이렇게 우회합니다:

```
상자 폭을 자로 잰다  →  그 폭에 해당하는 stroke 로 '명령'한다
                     →  빈 그리퍼도 같은 stroke 로 닫아 기준선을 잡는다
                     →  두 측정의 손가락 위치가 같아진다 ✓
```

즉 **"완전히 닫아라"가 아니라 "이 위치까지 닫아라"** 로 바꿉니다.

## 1. 연결

```bash
./scripts/robot_up.sh
```

In [ ]:
import json
import statistics
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import rclpy

import DR_init

ROBOT_ID, ROBOT_MODEL = "dsr01", "e0509"
DR_init.__dsr__id = ROBOT_ID
DR_init.__dsr__model = ROBOT_MODEL

rclpy.init()
node = rclpy.create_node("grasp_v2", namespace=ROBOT_ID)
DR_init.__dsr__node = node          # ← DSR_ROBOT2 import 보다 먼저여야 함

from DSR_ROBOT2 import (
    posx, movel, wait, DR_BASE,
    set_robot_mode, get_robot_mode, get_current_posx,
    set_tool, set_tcp, get_tool, get_tcp,
    read_data_rt,
    ROBOT_MODE_MANUAL, ROBOT_MODE_AUTONOMOUS,
)
from dsr_gripper import gripper_cmd

DATA = Path.cwd()          # json 저장 위치 (노트북과 같은 폴더)
print("연결 완료. 모드:", get_robot_mode(), "(1=autonomous)")

## 2. 모드 전환 — 확인하고 넘어갑니다

`set_robot_mode()` 는 **비동기**입니다. 요청만 하고 바로 다음 줄로 가면
아직 전환 전이라 명령이 조용히 실패합니다. (실제로 겪은 버그입니다)

**반드시 폴링해서 확인**한 뒤 진행합니다.

In [ ]:
def set_mode(mode, timeout=10.0):
    '''모드를 바꾸고 실제로 바뀔 때까지 기다린다. 안 바뀌면 예외.'''
    want = int(mode)
    if get_robot_mode() == want:
        return want
    set_robot_mode(want)
    t0 = time.time()
    while time.time() - t0 < timeout:
        if get_robot_mode() == want:
            name = "MANUAL" if want == 0 else "AUTONOMOUS"
            print(f"  모드 → {name} ({time.time()-t0:.1f}s)")
            return want
        time.sleep(0.3)
    raise RuntimeError(f"모드 전환 실패 (현재 {get_robot_mode()}, 원한 값 {want})")

## 3. 툴 등록 — 안 하면 무게 측정이 아예 안 됩니다

컨트롤러가 팔 끝에 뭐가 달렸는지 모르면 중력 모델이 틀리고, 무게를 못 잽니다.
**manual 모드에서만** 등록됩니다.

> `cog` / TCP 는 **근사값**입니다. 그리퍼 도면으로 실측해서 채우세요.
> 무게 0.5 kg 은 ROBOTIS 공식 스펙이라 정확합니다.

In [ ]:
from dsr_msgs2.srv import ConfigCreateTool, ConfigCreateTcp

TOOL_NAME, TCP_NAME = "rh_p12_rn", "rh_p12_rn_tcp"
TOOL_WEIGHT = 0.5                            # kg — ROBOTIS 스펙 (확실)
TOOL_COG = [0.0, 0.0, 60.0]                  # mm — TODO(실측)
TCP_POS = [0.0, 0.0, 150.0, 0.0, 0.0, 0.0]   # mm — TODO(실측)


def _call(cli, req, what, timeout=10.0):
    '''서비스 호출 + 성공 여부 확인. 실패하면 예외.'''
    if not cli.wait_for_service(timeout_sec=5.0):
        raise RuntimeError(f"{what}: 서비스 없음")
    fut = cli.call_async(req)
    rclpy.spin_until_future_complete(node, fut, timeout_sec=timeout)
    res = fut.result()
    if res is None:
        raise RuntimeError(f"{what}: 응답 없음 (timeout)")
    if not getattr(res, "success", True):
        raise RuntimeError(f"{what}: 거부됨 (success=False). manual 모드인지 확인하세요")
    return res


def register_tool():
    ns = f"/{ROBOT_ID}/dsr_controller2"
    tool_cli = node.create_client(ConfigCreateTool, f"{ns}/tool/config_create_tool")
    tcp_cli = node.create_client(ConfigCreateTcp, f"{ns}/tcp/config_create_tcp")

    set_mode(ROBOT_MODE_MANUAL)              # ← 등록은 manual 에서만

    r = ConfigCreateTool.Request()
    r.name, r.weight, r.cog, r.inertia = TOOL_NAME, TOOL_WEIGHT, TOOL_COG, [0.0] * 6
    _call(tool_cli, r, "config_create_tool")

    r = ConfigCreateTcp.Request()
    r.name, r.pos = TCP_NAME, TCP_POS
    _call(tcp_cli, r, "config_create_tcp")

    set_tool(TOOL_NAME)
    set_tcp(TCP_NAME)

    set_mode(ROBOT_MODE_AUTONOMOUS)          # ← 그리퍼는 autonomous 에서만 동작

    tool, tcp = get_tool(), get_tcp()
    if not tool or not tcp:
        raise RuntimeError(f"툴 등록 실패 (tool={tool!r}, tcp={tcp!r})")
    print(f"✓ tool={tool}  tcp={tcp}")


register_tool()

## 4. 그리퍼 — 위치를 '지정'해서 닫습니다

| 값 | 의미 |
|---|---|
| `position` | `0` = 완전 닫힘 ~ `750` = 완전 열림 |
| `current` | **파지 힘** (뉴턴이 아니라 레지스터 raw 값) |

**v1 의 가장 큰 버그가 뉴턴과 `current` 를 섞은 것**이었습니다. 여기서는 뉴턴을 아예 안 씁니다.

기준선을 맞추려면 **손가락 위치를 알아야** 하므로, "완전히 닫아라"가 아니라
**"이 stroke 까지 닫아라"** 로 명령합니다.

In [ ]:
STROKE_CLOSED, STROKE_OPEN = 0, 750
OPEN_WIDTH_MM = 109.0      # ROBOTIS 스펙. TODO(검증): 자로 재서 확인하세요

CURRENT_MIN = 50           # 이보다 낮으면 아예 못 잡음
CURRENT_MAX = 500          # TODO(실험): 상자가 찌그러지기 시작하는 값
CURRENT_PROBE = 100        # 무게 잴 때 '살짝' 잡는 힘. TODO(실험): 물체마다 적정값 확인

SQUEEZE_MM = 3.0           # 상자 폭보다 이만큼 더 조여서 확실히 문다


def stroke_from_width_mm(w_mm):
    '''개구 폭(mm) → stroke(0~750).'''
    s = round(w_mm / OPEN_WIDTH_MM * STROKE_OPEN)
    return int(max(STROKE_CLOSED, min(STROKE_OPEN, s)))


def width_mm_from_stroke(s):
    return max(0, min(STROKE_OPEN, s)) / STROKE_OPEN * OPEN_WIDTH_MM


def clamp_current(c, context=""):
    '''안전 범위로 자르되, MAX 초과는 '위험 신호'이므로 크게 경고한다.

    MIN 미만  : 계산값이 너무 약함 → 올림 (정보)
    MAX 초과  : 이 물체는 지금 설정으로 못 버틸 수 있음 → 경고 (위험)
    '''
    c = round(c)
    if c > CURRENT_MAX:
        print(f"  ⚠️ [{context}] 필요 힘 {c} 이 CURRENT_MAX({CURRENT_MAX})를 넘습니다.")
        print(f"     → 이 물체는 지금 설정으로 놓칠 수 있습니다. 상자가 찌그러지지 않는")
        print(f"        선에서 CURRENT_MAX 를 올리거나, 이 물체는 포기해야 합니다.")
        return CURRENT_MAX, "over_max"
    if c < CURRENT_MIN:
        print(f"  · [{context}] 계산값 {c} → 최소값 {CURRENT_MIN} 으로 올림")
        return CURRENT_MIN, "under_min"
    return int(c), "ok"


def grip_at(stroke, current, context="grip"):
    '''지정한 stroke 까지, 지정한 힘으로 닫는다.'''
    cur, _ = clamp_current(current, context)
    gripper_cmd(int(stroke), current=cur)
    wait(1)
    return cur


def release():
    gripper_cmd(STROKE_OPEN, current=CURRENT_MIN)
    wait(1)


print(f"stroke 0~{STROKE_OPEN} ↔ 0~{OPEN_WIDTH_MM}mm   current {CURRENT_MIN}~{CURRENT_MAX}")

## 5. 무게 측정

### 기준선은 stroke 마다 다릅니다

빈 그리퍼를 여러 stroke 에서 닫고 토크를 재서 **기준선 테이블**을 만듭니다.
측정할 때는 그 상자의 stroke 에 해당하는 기준선을 보간해서 씁니다.

### 캘리브레이션은 다중점 회귀입니다

v1 은 1점으로 비율만 구했습니다(원점 통과 가정). 그건 순환논리였습니다.
여기서는 **여러 무게로 `질량 = a·Δτ + b` 를 회귀**하고, **선형성을 확인**합니다.

In [ ]:
TORQUE_SAMPLES = 40          # 0.05초 간격 → 약 2초
WEIGHT_JOINT = 2             # J3 (0-based). 이 자세에서 가장 크게 반응한 관절
CONF_SIGMA = 3.0             # 신호가 노이즈의 이 배수를 넘어야 신뢰

BASELINE_PATH = DATA / "baseline_table.json"
CALIB_PATH = DATA / "weight_calib.json"


def motor_torque(n=TORQUE_SAMPLES):
    '''actual_motor_torque 의 (중앙값, 표준편차) — 전체 관절.'''
    rows = []
    for _ in range(n):
        d = read_data_rt()
        if d is not None:
            rows.append(list(d.actual_motor_torque))
        time.sleep(0.05)
    if not rows:
        raise RuntimeError("read_data_rt 가 데이터를 주지 않습니다")
    cols = list(zip(*rows))
    return ([statistics.median(c) for c in cols],
            [statistics.pstdev(c) if len(c) > 1 else 0.0 for c in cols])


class WeightSensor:
    '''모터 토크로 무게를 잰다.

    기준선은 stroke 에 의존하므로 테이블로 관리하고 보간한다.
    캘리브레이션은 여러 무게로 mass = a·Δτ + b 를 회귀한다.
    '''

    def __init__(self):
        self.baselines = {}      # {stroke: (torque_med, torque_sd)}  ← 전체 관절
        self.points = []         # [{stroke, dtau, mass_kg}]  캘리브레이션 점들
        self.a = self.b = None   # mass = a·dtau + b
        self._load()

    # ── 저장/로드 ───────────────────────────────────────
    def _load(self):
        if BASELINE_PATH.exists():
            raw = json.loads(BASELINE_PATH.read_text())
            self.baselines = {int(k): (v["med"], v["sd"]) for k, v in raw.items()}
            print(f"기준선 테이블 로드: stroke {sorted(self.baselines)}")
        if CALIB_PATH.exists():
            d = json.loads(CALIB_PATH.read_text())
            self.points, self.a, self.b = d["points"], d["a"], d["b"]
            print(f"캘리브레이션 로드: mass = {self.a:.1f}·Δτ + {self.b:+.3f}  ({len(self.points)}점)")

    def _save_baselines(self):
        BASELINE_PATH.write_text(json.dumps(
            {str(k): {"med": v[0], "sd": v[1]} for k, v in self.baselines.items()},
            indent=2))

    def _save_calib(self):
        CALIB_PATH.write_text(json.dumps(
            {"points": self.points, "a": self.a, "b": self.b}, indent=2, ensure_ascii=False))

    # ── 기준선 ─────────────────────────────────────────
    def take_baseline(self, stroke):
        '''빈 그리퍼를 그 stroke 까지 닫고 기준선을 잡는다. 픽 자세에서 실행하세요.'''
        release()
        grip_at(stroke, CURRENT_PROBE, context="baseline")
        time.sleep(1)
        med, sd = motor_torque()
        self.baselines[int(stroke)] = (med, sd)
        self._save_baselines()
        j = WEIGHT_JOINT
        print(f"  stroke {stroke:>3} → J{j+1} = {med[j]:+.3f} Nm  (노이즈 {sd[j]:.3f})")
        return med

    def build_baseline_table(self, strokes=(0, 150, 300, 450, 600)):
        '''여러 stroke 에서 기준선을 잡는다. 상자 크기가 달라도 대응할 수 있게.'''
        print(f"기준선 테이블 수집 — 그리퍼를 비우고 실행하세요 ({len(strokes)}점)")
        for s in strokes:
            self.take_baseline(s)
        print("완료. 상자를 잴 때는 그 상자의 stroke 로 보간해서 씁니다.")

    def baseline_at(self, stroke):
        '''stroke 에서의 기준선을 선형 보간. 테이블 밖이면 가장 가까운 값.'''
        if not self.baselines:
            raise RuntimeError("기준선이 없습니다. build_baseline_table() 을 먼저 실행하세요")
        ks = sorted(self.baselines)
        j = WEIGHT_JOINT
        if stroke <= ks[0]:
            return self.baselines[ks[0]][0][j], self.baselines[ks[0]][1][j]
        if stroke >= ks[-1]:
            return self.baselines[ks[-1]][0][j], self.baselines[ks[-1]][1][j]
        for lo, hi in zip(ks, ks[1:]):
            if lo <= stroke <= hi:
                t = (stroke - lo) / (hi - lo)
                m = self.baselines[lo][0][j] * (1 - t) + self.baselines[hi][0][j] * t
                s = max(self.baselines[lo][1][j], self.baselines[hi][1][j])
                return m, s

    # ── 측정 ───────────────────────────────────────────
    def raw_delta(self, stroke):
        '''지금 물고 있는 물체의 Δτ (기준선 대비). (dtau, noise, 전체관절Δ) 반환.'''
        med, sd = motor_torque()
        base_med, base_sd = self.baseline_at(stroke)
        j = WEIGHT_JOINT
        dtau = med[j] - base_med
        noise = max(sd[j], base_sd)
        # 전체 관절 Δ 도 같이 남긴다 (나중에 다중관절 결합을 데이터로 판단하려고)
        full = [med[i] - self.baselines[min(self.baselines, key=lambda k: abs(k - stroke))][0][i]
                for i in range(6)]
        return dtau, noise, full

    def measure(self, stroke):
        '''질량(kg)과 신뢰 여부를 함께 반환. → (mass_kg, ok)

        ok=False 면 신호가 노이즈에 묻힌 것이므로 그 값을 쓰면 안 됩니다.
        '''
        if self.a is None:
            raise RuntimeError("캘리브레이션이 없습니다. fit_calibration() 을 먼저 실행하세요")
        dtau, noise, _ = self.raw_delta(stroke)
        ok = abs(dtau) >= CONF_SIGMA * noise
        mass = self.a * abs(dtau) + self.b
        mass = max(0.0, mass)
        if ok:
            print(f"  Δτ = {dtau:+.3f} Nm (노이즈 {noise:.3f}) → {mass*1000:.0f} g")
        else:
            print(f"  ⚠️ Δτ({dtau:+.3f})가 노이즈({noise:.3f})의 {CONF_SIGMA}배 미만입니다.")
            print(f"     물체가 매달려 있지 않거나 너무 가볍습니다. 이 값을 쓰지 마세요.")
        return mass, ok

    # ── 캘리브레이션 (다중점 회귀) ─────────────────────
    def add_calib_point(self, known_mass_kg, stroke):
        '''무게를 아는 물체를 물린 상태에서 실행. 여러 무게로 3점 이상 모으세요.'''
        dtau, noise, full = self.raw_delta(stroke)
        if abs(dtau) < CONF_SIGMA * noise:
            print(f"  ⚠️ Δτ({dtau:+.3f})가 노이즈({noise:.3f})에 묻힙니다. 물체가 매달려 있나요?")
        self.points.append({
            "mass_kg": float(known_mass_kg), "dtau": float(dtau),
            "stroke": int(stroke), "noise": float(noise),
            "full_dtau": [float(v) for v in full],
            "t": datetime.now().isoformat(timespec="seconds"),
        })
        self._save_calib() if self.a is not None else CALIB_PATH.write_text(
            json.dumps({"points": self.points, "a": None, "b": None}, indent=2, ensure_ascii=False))
        print(f"  [{len(self.points)}] {known_mass_kg*1000:.0f} g → Δτ {dtau:+.3f} Nm  (저장됨)")

    def fit_calibration(self):
        '''질량 = a·|Δτ| + b 를 최소자승 회귀 + 선형성 확인 (LOO-CV).'''
        n = len(self.points)
        if n < 3:
            print(f"캘리브레이션 점이 {n}개 — 최소 3개 필요합니다 (선형성 확인용)")
            return None

        x = np.array([abs(p["dtau"]) for p in self.points])
        y = np.array([p["mass_kg"] for p in self.points])
        A = np.vstack([x, np.ones_like(x)]).T
        (self.a, self.b), *_ = np.linalg.lstsq(A, y, rcond=None)

        pred = self.a * x + self.b
        resid = pred - y
        rmse = float(np.sqrt(np.mean(resid ** 2)))
        r2 = 1 - np.sum(resid ** 2) / max(np.sum((y - y.mean()) ** 2), 1e-12)

        print(f"\n  질량 = {self.a:.1f}·|Δτ| {self.b:+.4f}   (kg)")
        print(f"  데이터 {n}점 · RMSE {rmse*1000:.1f} g · R² {r2:.4f}\n")
        print(f"  {'실제(g)':>8} {'예측(g)':>8} {'오차(g)':>8}   Δτ")
        for p, pr in zip(self.points, pred):
            e = (pr - p["mass_kg"]) * 1000
            flag = "  ⚠" if abs(e) > 0.15 * p["mass_kg"] * 1000 else ""
            print(f"  {p['mass_kg']*1000:>8.0f} {pr*1000:>8.0f} {e:>+8.0f}   {p['dtau']:+.3f}{flag}")

        # LOO-CV — 점이 적으니 held-out 대신 하나씩 빼고 검증
        if n >= 4:
            errs = []
            for i in range(n):
                m = [j for j in range(n) if j != i]
                Ai = np.vstack([x[m], np.ones(len(m))]).T
                (ai, bi), *_ = np.linalg.lstsq(Ai, y[m], rcond=None)
                errs.append(abs(ai * x[i] + bi - y[i]))
            print(f"\n  LOO-CV 평균오차 {np.mean(errs)*1000:.1f} g  (과적합 확인용)")

        if r2 < 0.95:
            print("\n  ⚠️ R² 가 낮습니다. 선형이 아닐 수 있습니다.")
            print("     원인 후보: 처짐에 따른 지렛대 변화 / 그리퍼 개도 차이 / 모터토크 포화")
            print("     → 산점도를 그려보고, 필요하면 stroke 별로 나눠서 회귀하세요.")

        self._save_calib()
        return self.a, self.b


scale = WeightSensor()

## 6. 픽 자세 티칭 — 파일로 저장됩니다

> ⚠️ **이 자세를 세션 내내 고정하세요.** 무게 측정이 자세에 의존합니다.
> 자세가 바뀌면 기준선 테이블과 캘리브레이션을 **처음부터 다시** 해야 합니다.

In [ ]:
PICK_PATH = DATA / "pick_pose.json"
PICK_POS = json.loads(PICK_PATH.read_text())["pick"] if PICK_PATH.exists() else None
if PICK_POS:
    print("저장된 픽 자세 로드:", PICK_POS)
else:
    print("저장된 픽 자세 없음 — 아래 두 셀로 티칭하세요")

In [ ]:
set_mode(ROBOT_MODE_MANUAL)
print("직접교시로 팔을 Pick 위치로 옮긴 뒤 다음 셀을 실행하세요.")

In [ ]:
_x, _ = get_current_posx()
PICK_POS = [round(v, 3) for v in _x]
PICK_PATH.write_text(json.dumps({"pick": PICK_POS,
                                 "t": datetime.now().isoformat(timespec="seconds")}, indent=2))
print("PICK_POS =", PICK_POS, "→ 저장됨 (커널 재시작해도 유지)")

set_mode(ROBOT_MODE_AUTONOMOUS)

## 7. 기준선 테이블 수집 (30분)

**그리퍼를 비우고**, 픽 자세에서 실행하세요.

In [ ]:
# 그리퍼에 아무것도 없어야 합니다
scale.build_baseline_table(strokes=(0, 150, 300, 450, 600))

## 8. 무게 캘리브레이션 (1시간) — **오늘의 첫 관문**

무게를 **저울로 잰** 물체 **3~4개**로 다중점 회귀합니다.

각 물체마다:
1. 자로 폭을 잰다 → `stroke = stroke_from_width_mm(폭)`
2. 그 stroke 로 물체를 문다
3. `scale.add_calib_point(질량kg, stroke)`

전부 모았으면 `scale.fit_calibration()`.

> **R² < 0.95 면 선형이 아닙니다.** 그때는 원인을 찾아야 합니다 (셀 출력이 후보를 알려줍니다).

In [ ]:
# 물체 하나마다 이 셀을 실행하세요
KNOWN_MASS_KG = 0.174      # 저울로 잰 값
WIDTH_MM = 70.0            # 자로 잰 폭

stroke = stroke_from_width_mm(WIDTH_MM - SQUEEZE_MM)
release()
input(f"물체를 그리퍼 사이에 놓고 Enter (stroke {stroke} 로 뭅니다)")
grip_at(stroke, CURRENT_PROBE, context="calib")
time.sleep(1)

scale.add_calib_point(KNOWN_MASS_KG, stroke)

In [ ]:
# 3~4개 모았으면 실행
scale.fit_calibration()

## 9. 파지 강도 계산

물리식이 **형태**를 줍니다: `F ≥ k·M·g / (2·μ)` → **필요한 힘은 무게에 비례**.

`current` 는 힘에 대략 비례하므로:

```
current = A · 질량(kg) + B
```

`A`, `B` 는 **실험으로** 구합니다. **뉴턴을 거치지 않습니다** — N ↔ current 변환은 알 수 없고,
알 필요도 없습니다.

In [ ]:
SAFETY = 1.2          # 안전 계수. 최소값에 딱 맞추면 실제로는 미끄러집니다
FIXED_CURRENT = 250   # 데이터가 없거나 무게를 못 믿을 때 쓰는 보수적 값


class GraspPlanner:
    def __init__(self):
        self.a = self.b = None

    def current_for(self, mass_kg=None, trustworthy=True):
        '''파지 current 를 정한다. (current, mode, status) 반환.

        무게를 못 믿거나(trustworthy=False) 계수가 없으면 fixed 로 자동 폴백.
        '''
        if self.a is None or mass_kg is None or not trustworthy:
            why = ("계수 없음" if self.a is None
                   else "무게 없음" if mass_kg is None else "무게 신뢰 불가")
            print(f"  → fixed 모드로 폴백 ({why}): current {FIXED_CURRENT}")
            c, st = clamp_current(FIXED_CURRENT, "fixed")
            return c, "fixed", st

        raw = (self.a * mass_kg + self.b) * SAFETY
        c, st = clamp_current(raw, "measured")
        return c, "measured", st

    def fit(self, trials):
        '''accept 된 (질량, 최소성공 current) 로 A, B 회귀 + LOO-CV.'''
        ok = [t for t in trials if t.get("accepted")]
        if len(ok) < 2:
            print(f"accept 된 데이터가 {len(ok)}개 — 2개 이상 필요합니다")
            return None
        m = np.array([t["mass_kg"] for t in ok])
        c = np.array([float(t["current"]) for t in ok])
        A = np.vstack([m, np.ones_like(m)]).T
        (self.a, self.b), *_ = np.linalg.lstsq(A, c, rcond=None)

        pred = self.a * m + self.b
        resid = pred - c
        rmse = float(np.sqrt(np.mean(resid ** 2)))
        r2 = 1 - np.sum(resid ** 2) / max(np.sum((c - c.mean()) ** 2), 1e-12)
        print(f"\n  current = {self.a:.0f}·M {self.b:+.0f}   (데이터 {len(ok)}점)")
        print(f"  RMSE {rmse:.1f} · R² {r2:.4f}")

        if len(ok) >= 4:
            errs = []
            for i in range(len(ok)):
                k = [j for j in range(len(ok)) if j != i]
                Ai = np.vstack([m[k], np.ones(len(k))]).T
                (ai, bi), *_ = np.linalg.lstsq(Ai, c[k], rcond=None)
                errs.append(abs(ai * m[i] + bi - c[i]))
            print(f"  LOO-CV 평균오차 {np.mean(errs):.1f} current")

        if r2 < 0.9:
            print("  ⚠️ R² 가 낮습니다. 판정이 애매했던 시도를 다시 보세요.")
        return self.a, self.b


planner = GraspPlanner()

## 10. 파지 시퀀스 — 들어올린 뒤 확인까지

```
① 상자 폭에 맞춰 살짝 집는다 → ② 무게를 잰다 → ③ 힘을 계산한다
                              → ④ 그 힘으로 다시 조인다 → ⑤ 들어올린다
                              → ⑥ 다시 재서 '아직 들고 있는지' 확인   ← 새로 추가
```

⑥ 이 없으면 **이동 중에 떨어뜨려도 모릅니다.** 정지 상태보다 이동 중(가속·감속)이
더 큰 힘을 요구하므로 오히려 더 위험한 구간입니다.

In [ ]:
LIFT_MM = 100.0
SPEED_L = 50


def move_to(pose, vel=SPEED_L):
    movel(posx(list(pose)), vel=vel, acc=vel, ref=DR_BASE)
    wait(1)


def pick_with_weight_feedback(width_mm, pick_pos=None, place_pos=None):
    '''살짝 집기 → 무게 재기 → 힘 계산 → 다시 조이기 → 들어서 확인 → 옮기기.'''
    pick_pos = pick_pos or PICK_POS
    stroke = stroke_from_width_mm(width_mm - SQUEEZE_MM)

    # ① 접근 + 살짝 집기
    release()
    move_to(pick_pos)
    grip_at(stroke, CURRENT_PROBE, context="probe")
    time.sleep(1)

    # ② 무게 측정 (픽 자세 그대로)
    mass, ok = scale.measure(stroke)

    # ③ 파지력 계산 — 무게를 못 믿으면 자동으로 fixed 폴백
    cur, mode, status = planner.current_for(mass if ok else None, trustworthy=ok)
    print(f"  질량 {mass*1000:.0f} g ({'신뢰' if ok else '불신'}) → current {cur} [{mode}]")

    # ④ 그 힘으로 다시 조인다
    grip_at(stroke, cur, context="grip")
    time.sleep(1)

    # ⑤ 들어올린다
    lifted = list(pick_pos)
    lifted[2] += LIFT_MM
    move_to(lifted)

    # ⑥ 아직 들고 있나? — 놓쳤으면 여기서 잡힙니다
    held, held_ok = scale.measure(stroke)
    if not held_ok or held < mass * 0.5:
        print(f"  ❌ 물체를 놓쳤습니다! (집을 때 {mass*1000:.0f}g → 지금 {held*1000:.0f}g)")
        if status == "over_max":
            print("     CURRENT_MAX 에 걸렸습니다. 이 물체는 지금 설정으로 못 듭니다.")
        return {"mass_kg": mass, "current": cur, "mode": mode, "held": False}

    print(f"  ✓ 들고 있습니다 ({held*1000:.0f}g)")

    if place_pos is not None:
        move_to(place_pos)
        release()
        up = list(place_pos)
        up[2] += LIFT_MM
        move_to(up)

    return {"mass_kg": mass, "current": cur, "mode": mode, "held": True}

## 11. 실험 루프 — `A`, `B` 를 구하는 데이터를 만든다

**오늘의 본 게임입니다.** 계수가 없으면 `measured` 모드가 안 켜집니다.

상자 하나마다:

```python
try_current(150, width_mm=70)    # 낮은 값부터
reject(0)                         # 미끄러졌으면
try_current(200, width_mm=70)
accept(1)                         # 버텼으면
```

**"버틴 것 중 가장 낮은 `current`" 를 accept 하세요.** 아무 성공값이나 넣으면 회귀가 과대평가됩니다.

무게를 바꿔가며 **최소 3종** 반복하세요.

> 상자가 **찌그러지기 시작하는 `current`** 도 `note` 에 남기세요 → `CURRENT_MAX` 를 그 아래로.

In [ ]:
TRIALS_PATH = DATA / "grasp_trials_v2.json"
trials = json.loads(TRIALS_PATH.read_text()) if TRIALS_PATH.exists() else []


def _save():
    TRIALS_PATH.write_text(json.dumps(trials, indent=2, ensure_ascii=False))


def try_current(current, width_mm, pick_pos=None, note=""):
    '''그 힘으로 집어 들어본다. 무게·폭·시각이 함께 기록된다.'''
    pick_pos = pick_pos or PICK_POS
    stroke = stroke_from_width_mm(width_mm - SQUEEZE_MM)

    release()
    move_to(pick_pos)

    grip_at(stroke, CURRENT_PROBE, context="probe")   # 먼저 살짝 집어 무게를 잰다
    time.sleep(1)
    mass, ok = scale.measure(stroke)

    cur = grip_at(stroke, current, context="trial")    # 시험할 힘으로 다시 조인다
    time.sleep(1)

    lifted = list(pick_pos)
    lifted[2] += LIFT_MM
    move_to(lifted)

    t = {"index": len(trials), "mass_kg": mass, "mass_ok": ok,
         "width_mm": width_mm, "stroke": stroke, "current": int(cur),
         "accepted": None, "note": note,
         "t": datetime.now().isoformat(timespec="seconds")}
    trials.append(t)
    _save()

    print(f"\n[{t['index']}] 질량 {mass*1000:.0f} g · 폭 {width_mm}mm · current {cur}")
    print("  → 미끄러졌나요? accept(i) / reject(i) 로 판정하세요.")
    return t["index"]


def review():
    print(f"{'idx':>3} | {'질량(g)':>8} | {'폭':>5} | {'current':>7} | {'판정':>8} | note")
    for t in trials:
        v = {True: "버팀", False: "미끄러짐", None: "-"}[t["accepted"]]
        w = "" if t.get("mass_ok", True) else " ⚠무게불신"
        print(f"{t['index']:>3} | {t['mass_kg']*1000:>8.0f} | {t['width_mm']:>5.0f} | "
              f"{t['current']:>7} | {v:>8} | {t['note']}{w}")


def accept(i, note=""):
    if not trials[i].get("mass_ok", True):
        print(f"  ⚠️ [{i}] 은 무게를 신뢰할 수 없는 시도입니다. 회귀가 오염됩니다.")
    trials[i]["accepted"] = True
    if note:
        trials[i]["note"] = note
    _save()
    print(f"✅ [{i}] 버팀 — 회귀에 사용됩니다")


def reject(i, note=""):
    trials[i]["accepted"] = False
    if note:
        trials[i]["note"] = note
    _save()
    print(f"❌ [{i}] 미끄러짐 — 기록만 남김")


print(f"이전 시도 {len(trials)}건 로드됨")

In [ ]:
# 다 모았으면 회귀
review()
planner.fit(trials)

## 12. 실제 파지

```python
pick_with_weight_feedback(width_mm=70)                  # 무게 기반 동적 파지
pick_with_weight_feedback(width_mm=70, place_pos=...)   # 옮기기까지
```

## 백로그 (오늘 안 함)

- **다중 관절 결합** — 지금은 J3만 씁니다. `full_dtau` 를 이미 기록하고 있으니,
  데이터가 쌓이면 J2+J3 결합으로 SNR 을 올릴 수 있는지 **데이터로 판단**하면 됩니다.
- **점진적 그립 + 실시간 슬립 감지** — 제약이 있습니다:
  - 그리퍼 **현재 위치를 읽을 수 없습니다** (`GripperCmd.srv` = 제어 전용, 피드백 없음).
    쓰려면 `gripper_service` 에 Modbus 읽기(FC03, Present Position=611)를 추가해야 합니다.
  - 슬립 신호로 **모터 토크**는 쓸 수 있지만, `motor_torque()` 가 **약 2초** 걸립니다.
    이동 중 실시간 감지엔 느립니다. 샘플 수를 줄이면 노이즈가 커지고요 — **트레이드오프를
    실측으로 정해야** 합니다.

In [ ]:
# pick_with_weight_feedback(width_mm=70)